# E50 --- a taxa que envelhece e a taxa que se paga

**A tentativa.** A secao anterior varreu a taxa e mediu o que cada escolha entrega. Falta o que
aquela conta pressupos: que a taxa fica fixa. Quem nunca a fixou --- a media acumulada, que da o
mesmo peso a todos os dias --- tem taxa 1/idade, e a taxa ENVELHECE a cada degrau.

**O que se mede.**

1. num mundo com seis degraus declarados, os dias para reaprender o nivel novo, em tres bracos:
   a media acumulada, o recomeco e a taxa fixa;
2. o preco da taxa fixa no mundo que nao muda: o piso estacionario medido contra a forma fechada.

**Convencoes** (AGENTS.md paragrafos 7 e 9): um experimento por caderno, parametros no topo
marcados com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E50_plasticidade.json,
figuras em .pdf e .png.

In [1]:
# <- brinque com: MUNDOS, DIAS, FATOR, DEGRAUS, HORIZONTE, TAXA, TAXAS_GRADE, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import frevolab
from frevolab import esquecimento, graficos, mudanca

MUNDOS = 20                    # os mundos do mesmo sorteio
DIAS = 6030                    # o mundo longo o bastante para seis degraus
FATOR = 2.0                    # cada degrau dobra a escala
DEGRAUS = (30, 1030, 2030, 3030, 4030, 5030)
HORIZONTE = 250                # a janela de leitura do capitulo
TAXA = esquecimento.TAXA_PADRAO
TAXAS_GRADE = (0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0)
SEMENTE = 117                  # E40..E49 usam 107..116
SIGMA = mudanca.SIGMA_PADRAO
VERDADE_UM = SIGMA * np.sqrt(2.0 / np.pi)   # a convencao do capitulo: o nivel e o |retorno| medio

print("frevolab %s | %d mundos de %d dias | taxas fixas em %d valores | semente %d"
      % (frevolab.VERSAO, MUNDOS, DIAS, len(TAXAS_GRADE), SEMENTE))

frevolab 0.1.0 | 20 mundos de 6030 dias | taxas fixas em 7 valores | semente 117


## Seis degraus, tres bracos

O dia do degrau e declarado, e a verdade de cada dia e a escala daquele trecho. O que se mede e o
erro na janela de leitura depois de cada degrau e os dias ate a tolerancia --- e o mundo em que o
dia nao chega e resultado, nao zero.

In [2]:
rng = np.random.default_rng(SEMENTE)
series = [np.abs(mudanca.degraus(DIAS, rng, SIGMA, fator=FATOR, quandon=DEGRAUS))
          for _ in range(MUNDOS)]
verdade = np.array([VERDADE_UM * FATOR ** sum(1 for q in DEGRAUS if d >= q)
                    for d in range(DIAS)])
caixa = esquecimento.plasticidade(series, DEGRAUS, verdade, horizonte=HORIZONTE, taxa=TAXA)

print("tolerancia do nivel %.0f%% | janela de leitura %d dias | taxa fixa %.3f"
      % (100 * caixa["tolerancia_nivel"], HORIZONTE, TAXA))
print("%-12s %-8s %10s %12s %14s" % ("braco", "degrau", "erro %", "dias", "sem volta"))
for braco, linhas in caixa["por_braco"].items():
    for dia, l in zip(DEGRAUS, linhas):
        print("%-12s %-8d %10.2f %12s %14d"
              % (braco, dia, 100 * l["erro_mediana"],
                 ("%.1f" % l["dias_mediana"]) if np.isfinite(l["dias_mediana"]) else "nunca",
                 l["mundos_sem_volta"]))

tolerancia do nivel 15% | janela de leitura 250 dias | taxa fixa 0.048
braco        degrau       erro %         dias      sem volta
acumulada    30            14.00         56.5              0
acumulada    1030          45.16        nunca             20
acumulada    2030          58.99        nunca             20
acumulada    3030          68.11        nunca             20
acumulada    4030          74.29        nunca             20
acumulada    5030          78.79        nunca             20
recomeco     30             7.39          5.0              0
recomeco     1030           7.89          5.0              0
recomeco     2030           5.60          3.0              0
recomeco     3030           7.31          3.5              0
recomeco     4030           6.66          4.0              0
recomeco     5030           5.59          4.0              0
exponencial  30            11.02         21.0              0
exponencial  1030          11.24         17.0              0
exponencial  2

In [3]:
# Figura 1: o erro na janela de leitura depois de cada degrau, nos tres bracos.
fig, eixo = plt.subplots(figsize=(9.0, 4.0))
cores = {"acumulada": "#b03a2e", "recomeco": "#1f4e79", "exponencial": "#c78f2c"}
for braco, linhas in caixa["por_braco"].items():
    erros = np.array([100 * l["erro_mediana"] for l in linhas])
    dispersoes = np.array([100 * l["erro_dispersao"] for l in linhas])
    eixo.errorbar(DEGRAUS, erros, yerr=dispersoes, fmt="o-", color=cores[braco],
                  capsize=4, label="a media %s" % braco)
eixo.axhline(100 * caixa["tolerancia_nivel"], color="#555555", ls="--", lw=1.4,
             label="a tolerancia do nivel: %.0f%%" % (100 * caixa["tolerancia_nivel"]))
eixo.set_xlabel("o dia do degrau")
eixo.set_ylabel("erro na janela de leitura (%)")
eixo.set_title("seis degraus: quem volta, e quem para", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E50_plasticidade", 1)
plt.close(fig)
print("figura E50_plasticidade_1 salva")

figura E50_plasticidade_1 salva


## O preco da taxa fixa: o piso do mundo que nao muda

A taxa fixa entrega a rapidez e cobra o piso. No mundo parado, o erro estacionario da media
exponencial sobe com a taxa --- e a forma e livre da lei dos dias: qualquer mundo de dias
independentes paga o mesmo fator.

In [4]:
rng_parado = np.random.default_rng(SEMENTE + 1)
piso_medido, piso_fechado, dias_por_taxa = [], [], []
for taxa in TAXAS_GRADE:
    parados = [np.abs(rng_parado.normal(0.0, SIGMA, DIAS)) for _ in range(MUNDOS)]
    sigmas = [float(np.std(s, ddof=1)) for s in parados]
    erros = []
    for s in parados:
        est = esquecimento.exponencial(s, taxa)
        erros.append(float(np.std(est[DIAS // 2:], ddof=1)))   # a dispersao estacionaria
    piso_medido.append(float(np.median(erros)))
    piso_fechado.append(float(np.median([esquecimento.piso_exponencial(taxa, sx) for sx in sigmas])))
    com_degraus = [np.abs(mudanca.degraus(DIAS, rng_parado, SIGMA, fator=FATOR, quandon=DEGRAUS))
                   for _ in range(MUNDOS)]
    dias = []
    for s in com_degraus:
        est = esquecimento.exponencial(s, taxa)
        d = esquecimento.dias_ate_dentro(est, verdade, DEGRAUS[-1], esquecimento.TOLERANCIA)
        dias.append(d if np.isfinite(d) else HORIZONTE)
    dias_por_taxa.append(float(np.median(dias)))

print("%-8s %12s %12s %10s" % ("taxa", "piso medido", "forma fechada", "dias"))
for taxa, medido, fechado, d in zip(TAXAS_GRADE, piso_medido, piso_fechado, dias_por_taxa):
    print("%-8.3f %12.4f %12.4f %10.1f" % (taxa, medido, fechado, d))

taxa      piso medido forma fechada       dias
0.020          0.0006       0.0006       69.5
0.050          0.0010       0.0010       22.0
0.100          0.0014       0.0014        9.0
0.200          0.0020       0.0020        3.0
0.400          0.0030       0.0030        4.0
0.700          0.0044       0.0044        4.0
1.000          0.0060       0.0060        5.0


In [5]:
# Figura 2: o piso contra a forma fechada, e os dias ate a tolerancia contra a taxa.
fig, eixos = plt.subplots(1, 2, figsize=(10.0, 3.9))
eixos[0].plot(TAXAS_GRADE, piso_medido, "o", color="#1f4e79", label="o piso medido")
grade = np.linspace(0.01, 1.0, 60)
sigma_x = float(np.mean([np.std(np.abs(rng_parado.normal(0.0, SIGMA, DIAS)), ddof=1) for _ in range(5)]))
eixos[0].plot(grade, [esquecimento.piso_exponencial(t, sigma_x) for t in grade], "-",
              color="#c78f2c", lw=1.4, label="a forma fechada")
eixos[0].set_xlabel("a taxa fixa")
eixos[0].set_ylabel("piso do erro no mundo parado")
eixos[0].set_title("o piso sobe com a taxa", fontsize=10)
eixos[0].legend(fontsize=8)
eixos[1].plot(TAXAS_GRADE, dias_por_taxa, "s-", color="#1f4e79")
eixos[1].set_xlabel("a taxa fixa")
eixos[1].set_ylabel("dias ate a tolerancia (mediana)")
eixos[1].set_title("os dias caem com a taxa", fontsize=10)
fig.suptitle("a mesma taxa paga os dois lados", fontsize=10)
fig.tight_layout()
graficos.salvar(fig, "E50_plasticidade", 2)
plt.close(fig)
print("figura E50_plasticidade_2 salva")

figura E50_plasticidade_2 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md paragrafo 9).

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: as tres curvas contra a linha da tolerancia --- a da media acumulada subindo com o
   dia do degrau e cruzando a linha, e as outras duas abaixo dela em todos os degraus.
2. **Figura 2**: no painel esquerdo os pontos medidos sobre a curva fechada; no direito os dias
   caindo quando a taxa sobe.

In [6]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
acumulada = caixa["por_braco"]["acumulada"][-1]
recomeco = caixa["por_braco"]["recomeco"][-1]
exponencial = caixa["por_braco"]["exponencial"][-1]


razoes = [medido / fechado for medido, fechado in zip(piso_medido, piso_fechado) if fechado > 0.0]  # medido sobre a forma fechada
resultado = {
    "plast_mundos": MUNDOS,
    "plast_dias": DIAS,
    "plast_degraus": len(DEGRAUS),
    "plast_horizonte": HORIZONTE,
    "plast_taxa": round(TAXA, 3),
    "plast_tolerancia_pct": round(100 * caixa["tolerancia_nivel"], 1),
    "plast_erro_acumulada_pct": round(100 * acumulada["erro_mediana"], 3),
    "plast_erro_recomeco_pct": round(100 * recomeco["erro_mediana"], 3),
    "plast_erro_exponencial_pct": round(100 * exponencial["erro_mediana"], 3),
    "plast_dias_acumulada": (-1 if not np.isfinite(acumulada["dias_mediana"])
                             else round(acumulada["dias_mediana"], 1)),
    "plast_dias_recomeco": (-1 if not np.isfinite(recomeco["dias_mediana"])
                            else round(recomeco["dias_mediana"], 1)),
    "plast_dias_exponencial": (-1 if not np.isfinite(exponencial["dias_mediana"])
                               else round(exponencial["dias_mediana"], 1)),
    "plast_sem_volta_acumulada": int(acumulada["mundos_sem_volta"]),
    "plast_sem_volta_recomeco": int(recomeco["mundos_sem_volta"]),
    "plast_sem_volta_exponencial": int(exponencial["mundos_sem_volta"]),
    "plast_piso_medido_pct": round(100 * float(np.median(piso_medido)) / VERDADE_UM, 3),
    "plast_piso_fechado_pct": round(100 * float(np.median(piso_fechado)) / VERDADE_UM, 3),
    "plast_piso_razao": round(float(np.median(razoes)), 3),
    "plast_dias_taxa_baixa": round(dias_por_taxa[0], 1),
    "plast_dias_taxa_alta": round(dias_por_taxa[-1], 1),
}
caminho = Path("lab/resultados/E50_plasticidade.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "plast_mundos": 20,
 "plast_dias": 6030,
 "plast_degraus": 6,
 "plast_horizonte": 250,
 "plast_taxa": 0.048,
 "plast_tolerancia_pct": 15.0,
 "plast_erro_acumulada_pct": 78.787,
 "plast_erro_recomeco_pct": 5.588,
 "plast_erro_exponencial_pct": 11.079,
 "plast_dias_acumulada": -1,
 "plast_dias_recomeco": 4.0,
 "plast_dias_exponencial": 24.0,
 "plast_sem_volta_acumulada": 20,
 "plast_sem_volta_recomeco": 0,
 "plast_sem_volta_exponencial": 0,
 "plast_piso_medido_pct": 25.258,
 "plast_piso_fechado_pct": 25.29,
 "plast_piso_razao": 1.002,
 "plast_dias_taxa_baixa": 69.5,
 "plast_dias_taxa_alta": 5.0
}
